In [ ]:
import anndata as ad
import os
import re
import numpy as np
import squidpy as sq
import scanpy as sc
import harmonypy as hm
import umap

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

### Load processed nuclear protein intensity and RNA matrices

In [ ]:
# Load single-cell RNA cycleHCR data from RNA-spot-to-cell assignment
# RNA matrix of the 2nd brain
rna_df = pd.read_csv("./input/cell_by_transcript_gene_name_matrix2.csv", index_col=0)

In [ ]:
# Load preprocessed nuclear protein intensity matrix from Step_8_1
adata_1st_brain = ad.read_h5ad('input/adata_18_nuclear_46_prot_brain1.h5ad')
adata_2nd_brain = ad.read_h5ad('input/brain2_nuclear_int_18prot.h5ad')

## Make integrated UMAP of two brain sections using Harmony

In [ ]:
# make a matrix with the common nuclear markers in both brains

common_vars = adata_1st_brain.var_names.intersection(adata_2nd_brain.var_names)
adata_1 = adata_1st_brain[:, common_vars].copy()
adata_2 = adata_2nd_brain[:, common_vars].copy()

In [ ]:
adata_both = ad.concat([adata_1, adata_2],
                      join="outer",     # keep all features
                      label="dataset",  # new column in .obs
                      keys=["1st_brain", "2nd_brain"])  

In [ ]:
common_genes = adata_1.var_names.intersection(adata_2.var_names)
len(common_genes)

In [ ]:
adata_both.layers["counts"] = adata_both.X.copy()
sc.pp.normalize_total(adata_both, inplace=True)
sc.pp.log1p(adata_both)

In [ ]:
sc.tl.pca(adata_both, svd_solver="arpack")

In [ ]:
ho = hm.run_harmony(adata_both.obsm['X_pca'], adata_both.obs, 'dataset', theta=6, lamb = 0.3, sigma = 0.02, nclust=100, max_iter_harmony = 20, random_state=42)   # 
# theta (diversity penalty) Default: 2 Larger values (e.g. 6, 8, 12) are used when default cannot resolve batch effect.
# Harmony run takes ~30 mins for our dataset


# save Harmony-corrected embeddings
adata_both.obsm['X_pca_harmony'] = ho.Z_corr.T

In [ ]:
X_pca_1 = adata_both[adata_both.obs["dataset"]=="1st_brain"].obsm["X_pca_harmony"]
X_pca_2 = adata_both[adata_both.obs["dataset"]=="2nd_brain"].obsm["X_pca_harmony"]

In [ ]:

reducer = umap.UMAP(n_neighbors=5, min_dist=0.1, n_components=2, random_state=42)
reducer.fit(X_pca_2)
X_umap_1 = reducer.transform(X_pca_1)
X_umap_2 = reducer.transform(X_pca_2)

In [ ]:
adata_1.obsm["X_umap"] = X_umap_1
adata_2.obsm["X_umap"] = X_umap_2

adata_plot = ad.concat(
    [adata_1, adata_2],
    join="outer",
    label="dataset",
    keys=["1st brain section", "2nd brain section"],
    merge="unique"
)

adata_plot.obsm['X_pca_harmony'] = np.vstack([X_pca_1, X_pca_2])
adata_plot.obsm["X_umap"] = np.vstack([X_umap_1, X_umap_2])

# 7) Plot
sc.set_figure_params(figsize=(15, 15))
sc.pl.umap(adata_plot, color=["dataset"])

In [ ]:
# Save matrix to skip rerunning Harmony everytime

# adata_plot.write('harmony_ave_int_2_brains.h5ad')   

In [ ]:
# load result 
# change the path to load your result that was saved in the above cell
# or use this path to show the downloaded result from the paper
adata_plot= ad.read('input/adata_plot_harmony_ave_int_2_brains.h5ad')

In [ ]:
sc.set_figure_params(figsize=(15, 15))
sc.pl.umap(adata_plot, color=["dataset"], size=5, alpha=0.9)

# plt.savefig("harmony_umap.eps", format="eps", bbox_inches="tight")
plt.close()

In [ ]:
print("neighbors")
sc.pp.neighbors(adata_plot, n_neighbors=10, random_state=42, use_rep='X_pca_harmony')

In [ ]:
print("Leiden")
resolution = 0.8       # 1 -- 40 clusters
sc.tl.leiden(adata_plot, resolution=resolution, random_state=38)

In [ ]:
sc.set_figure_params(figsize=(15, 15))
sc.pl.umap(adata_plot, color=["leiden"], size=8)

In [ ]:
ad_1st = adata_plot[adata_plot.obs.dataset == '1st brain section']
ad_2nd = adata_plot[adata_plot.obs.dataset == '2nd brain section']

In [ ]:
sc.pl.umap(ad_1st, color=["leiden"],size=7)

In [ ]:
# Save clustering results.

# adata_plot.write('harmony_ave_int_2_brains.h5ad')   

# Only one of the replicates -- Brain Section 2
# ad_2nd.write('harmony_ave_int_Brain_2.h5ad')

In [ ]:
# Load stored clustering result

ad_2nd = ad.read_h5ad('input/harmony_clusters_2nd_brain.h5ad') 

In [ ]:
import matplotlib.patheffects as PathEffects

sc.set_figure_params(figsize=(15, 15))

# Plot UMAP with on-data labels
sc.pl.umap(ad_2nd, color="leiden", legend_loc="on data", size=7, show=False)

ax = plt.gca()

ax.set_frame_on(False)   # removes the box
ax.set_xticks([])        # removes x ticks
ax.set_yticks([])        # removes y ticks
ax.set_xlabel("")        # removes x-axis title
ax.set_ylabel("")        # removes y-axis title
ax.set_title("")  

for txt in ax.texts:
    txt.set_fontsize(20)
    txt.set_fontname("Arial")
    txt.set_path_effects([
        PathEffects.withStroke(linewidth=1, foreground='white')  # white outline
    ])

for coll in ax.collections:
    coll.set_rasterized(True)

# plt.savefig("umap_Brain_2.pdf", dpi=300, bbox_inches="tight", pad_inches=0)
plt.show()

In [ ]:
# rotate spatial coordinates for plotting the brain section

# 1. Backup original coordinates
original_coords = ad_2nd.obsm["spatial"].copy()

# 2. Rotate coordinates
theta = np.radians(-137)
rotation_matrix = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])
rotated_coords = original_coords @ rotation_matrix.T
ad_2nd.obsm["spatial_rotated"] = rotated_coords

In [ ]:
coords = ad_2nd.obsm['spatial_rotated']
selected_clusters = ['18', '13', '14', '21', '9','0', '11', '10', ]     # edit to show different clusters

# Ensure Leiden categories and colors
leiden_categories = ad_2nd.obs['leiden'].cat.categories
leiden_colors = ad_2nd.uns['leiden_colors']
cluster_colors = dict(zip(leiden_categories, leiden_colors))

bg_color = "#EEEEEE"

# Output folder
output_dir = "./plots"
os.makedirs(output_dir, exist_ok=True)
save_path = os.path.join(output_dir, "Glut_clusters.pdf")

# Create figure
plt.figure(figsize=(10, 10))

# Background
plt.scatter(coords[:, 0], coords[:, 1],
            c=[bg_color]*coords.shape[0], s=6, edgecolors='none', rasterized=True)

# Overlay clusters
for cluster in selected_clusters:
    mask = ad_2nd.obs['leiden'] == cluster
    plt.scatter(coords[mask, 0], coords[mask, 1],
                c=[cluster_colors[cluster]]*np.sum(mask),
                s=0.4, label=f"Cluster {cluster}", rasterized=True)

plt.gca().set_aspect('equal')
plt.gca().invert_yaxis()
plt.axis('off')

# Legend on the right without box
plt.legend(markerscale=2, bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)

plt.show()

## RNA feature of clusters generated using nuclear protein intensity

In [ ]:
# Match cell IDs
rna_df.index = rna_df.index.astype(str).str.strip()
common_cells = ad_2nd.obs_names.intersection(rna_df.index)
rna_df = rna_df.loc[common_cells]
adata_rna = ad.AnnData(
    X=rna_df.to_numpy(),
    obs=ad_2nd.obs.loc[common_cells].copy(),
    var=pd.DataFrame(index=rna_df.columns.astype(str))
)
adata_rna.obs_names = common_cells
adata_rna.var_names = rna_df.columns.astype(str)

for key in ad_2nd.obsm.keys():
    adata_rna.obsm[key] = ad_2nd.obsm[key][ad_2nd.obs_names.get_indexer(common_cells)]

adata_rna.uns = ad_2nd.uns.copy()

In [ ]:
adata_rna

In [ ]:
sc.pp.normalize_total(adata_rna, inplace=True)
sc.pp.log1p(adata_rna)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
sc.tl.rank_genes_groups(adata_rna, groupby="leiden", method="wilcoxon")

In [ ]:
result = adata_rna.uns['rank_genes_groups']
groups = result['names'].dtype.names

dat = pd.DataFrame({group + '_' + key[:1]: result[key][group] for group in groups for key in ['names', 'logfoldchanges','scores','pvals']})
# dat.to_csv("harmony_ave_int_cluster_genes.csv")

In [ ]:
sc.pl.rank_genes_groups(adata_rna, n_genes=15, fontsize=40)

In [ ]:
# construct high variation gene list for plotting

from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist

import warnings
warnings.simplefilter("ignore", category=pd.errors.PerformanceWarning)

# Convert 'leiden' to categorical if not already
adata_rna.obs['leiden'] = adata_rna.obs['leiden'].astype('category')

# remove small clusters with few cells if needed
keep_clusters = [str(i) for i in range(30)]
adata_filtered2 = adata_rna[adata_rna.obs['leiden'].isin(keep_clusters)].copy()

# Remove unused categories
adata_filtered2.obs['leiden'] = adata_filtered2.obs['leiden'].cat.remove_unused_categories()

sc.tl.rank_genes_groups(adata_filtered2, groupby="leiden", method="t-test")  # or 'wilcoxon'
marker_df2 = sc.get.rank_genes_groups_df(adata_filtered2, group=None)

In [ ]:
marker_genes_per_cluster = {}
marker_genes_per_cluster_filtered = {}

n_genes = 10           # only keep top number of genes with highest log fold change for each cluster

for cluster in marker_df2['group'].unique():
    cluster_df = marker_df2[marker_df2['group'] == cluster]
    
    # Only keep genes with logfoldchange > 0.6 for that cluster
    cluster_df = cluster_df[cluster_df['logfoldchanges'] > 0.6]

    # Take up to n_genes
    top_genes = cluster_df['names'].head(n_genes).tolist()
    
    marker_genes_per_cluster_filtered[cluster] = top_genes

# Non-repeating gene list
used_genes = set()
unique_marker_genes_filtered = {}
for cluster, gene_list in marker_genes_per_cluster_filtered.items():
    unique_genes = []
    for gene in gene_list:
        if gene not in used_genes:
            unique_genes.append(gene)
            used_genes.add(gene)
    unique_marker_genes_filtered[cluster] = unique_genes

# Flatten for plotting/export
unique_genes_to_plot = [gene for genes in unique_marker_genes_filtered.values() for gene in genes]

# Optional: print or make DataFrame
print("Unique marker genes:")
for cluster, genes in unique_marker_genes_filtered.items():
        print(f"  Cluster {cluster}: {genes}")

In [ ]:
# Reorder the gene-cluster matrix by grouping similar expression patterns

# Compute expression matrix for clustering
cluster_ids = adata_filtered2.obs['leiden'].cat.categories
gene_list = [gene for genes in unique_marker_genes_filtered.values() for gene in genes]

expr_matrix = []
for cluster in cluster_ids:
    cells = adata_filtered2[adata_filtered2.obs['leiden'] == cluster]
    mean_expr = cells[:, gene_list].X.mean(axis=0).flatten()
    expr_matrix.append(mean_expr)

expr_df = pd.DataFrame(expr_matrix, index=cluster_ids, columns=gene_list)

# Cluster clusters by gene expression
linkage_matrix = linkage(pdist(expr_df, metric='euclidean'), method='average')
ordered_clusters = expr_df.index[leaves_list(linkage_matrix)].tolist()

# Reorder cluster categories
adata_filtered2.obs['leiden'] = adata_filtered2.obs['leiden'].cat.reorder_categories(ordered_clusters)


unique_genes_to_plot = []
used_genes = set()

for cluster in ordered_clusters:
    cluster_genes = unique_marker_genes_filtered.get(cluster, [])
    for gene in cluster_genes:
        if gene not in used_genes:
            unique_genes_to_plot.append(gene)
            used_genes.add(gene)

var_names = {
    f"{cluster}": unique_marker_genes_filtered[cluster]
    for cluster in ordered_clusters
    if cluster in unique_marker_genes_filtered
}


In [ ]:
total_genes = sum(len(genes) for genes in var_names.values())
print(f"Total number of genes in plot: {total_genes}")

## Export to plot in R

In [ ]:
# List of marker genes 
genes_to_plot = unique_genes_to_plot

# Get cluster categories
clusters = adata_filtered2.obs['leiden'].cat.categories

# Compute mean expression per cluster
expr_matrix = []
for clust in clusters:
    cells = adata_filtered2[adata_filtered2.obs['leiden'] == clust]
    mean_expr = cells[:, genes_to_plot].X.mean(axis=0).flatten()
    expr_matrix.append(mean_expr)

expr_df = pd.DataFrame(expr_matrix, index=clusters, columns=genes_to_plot)

# Export expression matrix
# expr_df.to_csv("R_expression_matrix.csv")

In [ ]:
#  export z-score matrix

from scipy.stats import zscore     
expr_zscore = expr_df.apply(zscore, axis=0)
# expr_zscore.to_csv("R_expression_zscore.csv")

## Export nuclear protein intensities of each cluster to plot in R

In [ ]:
# read nuclear protein intensity matrix
proteins_to_plot = ad_2nd.var_names

# Get cluster categories
clusters = ad_2nd.obs['leiden'].cat.categories

# Compute mean expression per cluster
expr_matrix = []
for clust in clusters:
    cells = ad_2nd[ad_2nd.obs['leiden'] == clust]
    mean_expr = cells[:, proteins_to_plot].X.mean(axis=0).flatten()
    expr_matrix.append(mean_expr)

expr_df = pd.DataFrame(expr_matrix, index=clusters, columns=proteins_to_plot)

# Save
# expr_df.to_csv("R_nuclear_protein_matrix.csv")

In [ ]:
# Save z-scores

from scipy.stats import zscore
expr_zscore = expr_df.apply(zscore, axis=0)
# expr_zscore.to_csv("R_nuclear_protein_zscore_test.csv")